# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding 1:** "Cluster 5 (long-form, ~2,200 pages) shows the steepest average decline (~-28%) and the weakest engagement/scroll rate of all archetypes." **Methodology question:** is -28% a stable estimate for a cluster this size, or could a handful of extreme-decline pages be pulling the average down? (Check the median alongside the mean, not just the mean.)

**Finding 2:** "Cluster 3 (~980 pages) has the highest impressions and AI referral traffic of any cluster." **Methodology question:** this is the smallest, least stable cluster across reseeding (see below) — is this a genuinely distinct archetype, or a boundary effect between two nearby clusters?

In [1]:
# Path assumes this notebook lives in work/notebooks/ — adjust if you moved it.
DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"

import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

df = pd.read_csv(DATA_PATH)
df.loc[df["trend_direction"].isin(["flat","new"]), "trend_pct"] = 0.0
df.loc[df["avg_position"] == 0, "avg_position"] = np.nan
numeric_features = [
    "word_count","char_count","impressions_90d","clicks_90d","pageviews_90d",
    "sessions_90d","users_90d","engaged_sessions_90d","ai_sessions_90d",
    "scroll_events_90d","content_age_days","days_since_last_update","ctr",
    "avg_position","engagement_rate","scroll_rate","ai_traffic_pct","trend_pct",
]
X_num = df[numeric_features].copy()
for c in ["impressions_90d","clicks_90d","pageviews_90d","sessions_90d",
          "users_90d","engaged_sessions_90d","ai_sessions_90d","scroll_events_90d"]:
    X_num[c] = np.log1p(X_num[c])
X_num["word_count_was_missing"] = df["word_count"].isnull().astype(int)
X_num["char_count_was_missing"] = df["char_count"].isnull().astype(int)
X_num = X_num.fillna(X_num.median(numeric_only=True))
X_cat = pd.get_dummies(df[["trend_direction"]].fillna("unknown"), drop_first=True)
X = pd.concat([X_num, X_cat], axis=1).fillna(0)
Xs = StandardScaler().fit_transform(X)

df["cluster_seedA"] = KMeans(n_clusters=7, random_state=42, n_init=10).fit_predict(Xs)
df["cluster_seedB"] = KMeans(n_clusters=7, random_state=99, n_init=10).fit_predict(Xs)

print("Seed A cluster sizes:", df["cluster_seedA"].value_counts().sort_index().to_dict())
print("Seed B cluster sizes:", df["cluster_seedB"].value_counts().sort_index().to_dict())

Seed A cluster sizes: {0: 5160, 1: 2212, 2: 6495, 3: 12187, 4: 2706, 5: 100, 6: 1140}
Seed B cluster sizes: {0: 12152, 1: 1140, 2: 5157, 3: 6491, 4: 143, 5: 2211, 6: 2706}


## 2. My model under an honest split (before/after)

In [2]:
# The 'before/after' check for clustering is stability across reseeding, run above.
# Compare: cluster sizes should roughly match (just possibly relabeled) if the
# clustering is genuinely reflecting structure in the data, not seed noise.

The larger clusters (the moderate-traffic bulk, the champions, the low-demand group) reproduce closely across both seeds. The smallest cluster's boundary with its nearest neighbor shifts more between seeds — that's the honest limitation to report for Finding 2 above, not something to hide.

## 3. Leakage audit

In [3]:
# Confirm no product-decision fields or their bucketed versions made it into X
product_flags = ["health_score","priority_score","action_type","needs_ctr_fix","is_quick_win"]
print("Product flags present:", [c for c in product_flags if c in df.columns])
tier_leak = [c for c in X.columns if "tier" in c]
print("Tier columns leaked into features:", tier_leak)

Product flags present: []
Tier columns leaked into features: []


## 4. Claim rewrite

**Bold version (don't say this):** "Cluster 5 pages are failing and need immediate rewrites."

**Safe rewrite:** "In this 90-day snapshot, cluster 5 pages show the steepest observed average decline and the weakest observed engagement of the seven archetypes — this is directional and decision-support only, not a guarantee that a rewrite would reverse the trend."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words (observed / directional / decision-support), never causal or 'predicting Google'